# Rainfall Anomaly Detection — SARIMA Model
 
**Model:** SARIMA (Seasonal AutoRegressive Integrated Moving Average)  

**Dataset:** District-wise Rainfall in India (1901–2015) - data.gov.in

**Metrics:**
Confusion Matrix · Accuracy · Precision · Recall · F1-Score · Sensitivity · AUC · Error Rate


In [ ]:
#Install libraries
!pip install statsmodels scikit-learn matplotlib seaborn pandas numpy -q

In [ ]:
#Upload rainfall_india.csv
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

In [ ]:
#Import all libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay
)

# ── Plot style
plt.rcParams.update({
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})
BLUE   = "#1F4E79"
ORANGE = "#E76F51"
GREEN  = "#2D9D5F"
RED    = "#D62828"

print("All libraries imported successfully")

In [ ]:
#Load and preprocess dataset
df_raw = pd.read_csv("rainfall_india.csv")
df_raw.columns = df_raw.columns.str.strip().str.upper()

MONTH_COLS = ["JAN","FEB","MAR","APR","MAY","JUN",
              "JUL","AUG","SEP","OCT","NOV","DEC"]
MONTH_NUM  = {m: i+1 for i, m in enumerate(MONTH_COLS)}

df = (df_raw[["SUBDIVISION","YEAR"] + MONTH_COLS]
      .melt(id_vars=["SUBDIVISION","YEAR"],
            value_vars=MONTH_COLS,
            var_name="MONTH_NAME",
            value_name="RAINFALL"))

df["MONTH"]    = df["MONTH_NAME"].map(MONTH_NUM)
df["RAINFALL"] = pd.to_numeric(df["RAINFALL"], errors="coerce")
df = df.dropna(subset=["RAINFALL"]).reset_index(drop=True)
df["DATE"]     = pd.to_datetime({"year": df["YEAR"], "month": df["MONTH"], "day": 1})
df             = df.sort_values(["SUBDIVISION","DATE"]).reset_index(drop=True)

# ── Z-score anomaly labels
df["MEAN"] = df.groupby(["SUBDIVISION","MONTH"])["RAINFALL"].transform("mean")
df["STD"]  = df.groupby(["SUBDIVISION","MONTH"])["RAINFALL"].transform("std").fillna(1)
df["Z_SCORE"] = (df["RAINFALL"] - df["MEAN"]) / (df["STD"] + 1e-9)
df["ANOMALY"] = (df["Z_SCORE"].abs() > 2.0).astype(int)

# ── Pick subdivision with most records
TARGET_SUB = df["SUBDIVISION"].value_counts().index[0]
ts_df  = df[df["SUBDIVISION"] == TARGET_SUB].sort_values("DATE").reset_index(drop=True)
ts     = ts_df.set_index("DATE")["RAINFALL"]
labels = ts_df.set_index("DATE")["ANOMALY"]

print(f"Dataset loaded — {len(df):,} monthly records | 36 subdivisions")
print(f"Subdivision used for SARIMA: {TARGET_SUB}")
print(f"Time range: {ts.index.min().year} – {ts.index.max().year}  ({len(ts)} months)")
print(f"Anomaly rate: {df['ANOMALY'].mean()*100:.1f}%")

In [ ]:
#SARIMA model function
THRESHOLD = 2.0  # residual Z-score threshold for anomaly

def run_sarima(ts, labels, train_ratio, split_label):
    n        = len(ts)
    n_train  = int(n * train_ratio)
    ts_train = ts.iloc[:n_train]
    ts_test  = ts.iloc[n_train:]
    y_true   = labels.iloc[n_train:].values

    # Fit SARIMA(1,1,1)(1,1,1,12)
    model = SARIMAX(ts_train,
                    order=(1, 1, 1),
                    seasonal_order=(1, 1, 1, 12),
                    enforce_stationarity=False,
                    enforce_invertibility=False)
    fit      = model.fit(disp=False)
    forecast = fit.forecast(steps=len(ts_test))

    # Residual → Z-score → anomaly flag
    residuals  = ts_test.values - forecast.values
    res_z      = (residuals - residuals.mean()) / (residuals.std() + 1e-9)
    y_pred     = (np.abs(res_z) > THRESHOLD).astype(int)
    # Use |res_z| as a continuous anomaly score for AUC
    y_score    = np.abs(res_z)

    return {
        "split":    split_label,
        "n_train":  n_train,
        "n_test":   len(ts_test),
        "y_true":   y_true,
        "y_pred":   y_pred,
        "y_score":  y_score,
        "ts_test":  ts_test,
        "forecast": forecast,
        "res_z":    res_z,
    }

print("SARIMA function defined")

In [ ]:
#Run SARIMA for all 4 train-test splits
SPLITS = [(0.60,"60:40"), (0.70,"70:30"), (0.80,"80:20"), (0.90,"90:10")]
results = {}

print("Training SARIMA models...\n")
for ratio, label in SPLITS:
    print(f"  [{label}] fitting... ", end="", flush=True)
    r = run_sarima(ts, labels, ratio, label)
    results[label] = r
    acc  = accuracy_score(r["y_true"], r["y_pred"])
    print(f"done  →  Accuracy = {acc*100:.2f}%")

print("\nAll 4 splits complete")

In [ ]:
#Compute all metrics for every split
print("="*75)
print(f"{'Split':<10} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} "
      f"{'F1-Score':>10} {'AUC':>8} {'Error Rate':>12}")
print("="*75)

metrics_rows = []
for label, r in results.items():
    yt, yp, ys = r["y_true"], r["y_pred"], r["y_score"]

    acc   = accuracy_score(yt, yp)
    prec  = precision_score(yt, yp, zero_division=0)
    rec   = recall_score(yt, yp, zero_division=0)      # = Sensitivity
    f1    = f1_score(yt, yp, zero_division=0)
    err   = 1 - acc
    try:
        auc = roc_auc_score(yt, ys)
    except Exception:
        auc = float("nan")

    metrics_rows.append({
        "Split": label, "Accuracy": acc, "Precision": prec,
        "Recall / Sensitivity": rec, "F1-Score": f1,
        "AUC": auc, "Error Rate": err,
        "n_train": r["n_train"], "n_test": r["n_test"]
    })

    print(f"{label:<10} {acc*100:>9.2f}% {prec*100:>9.2f}% {rec*100:>9.2f}% "
          f"{f1*100:>9.2f}% {auc:>8.4f} {err*100:>11.2f}%")

print("="*75)
print("Note: Recall = Sensitivity (same metric, different names)")
metrics_df = pd.DataFrame(metrics_rows)
print("\nDataFrame preview:")
print(metrics_df[["Split","Accuracy","Precision","Recall / Sensitivity","F1-Score","AUC","Error Rate"]].to_string(index=False))

In [ ]:
#Confusion Matrix (all 4 splits)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("SARIMA — Confusion Matrix  (All Train-Test Splits)",
             fontsize=15, fontweight="bold", y=1.03)

for ax, (label, r) in zip(axes, results.items()):
    cm   = confusion_matrix(r["y_true"], r["y_pred"])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Normal", "Anomaly"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f"Split {label}\nTN={tn} FP={fp} FN={fn} TP={tp}",
                 fontsize=10, pad=8)

plt.tight_layout()
plt.savefig("01_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
#F1, Precision, Recall, Sensitivity
split_labels = [r["Split"] for r in metrics_rows]
f1_vals   = [r["F1-Score"]*100   for r in metrics_rows]
prec_vals = [r["Precision"]*100  for r in metrics_rows]
rec_vals  = [r["Recall / Sensitivity"]*100 for r in metrics_rows]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("SARIMA — F1-Score | Precision | Recall & Sensitivity",
             fontsize=14, fontweight="bold")

for ax, vals, title, color in zip(
        axes,
        [f1_vals, prec_vals, rec_vals],
        ["F1-Score (%)", "Precision (%)", "Recall = Sensitivity (%)"],
        [BLUE, ORANGE, GREEN]):

    bars = ax.bar(split_labels, vals, color=color, alpha=0.88, width=0.5, edgecolor="white")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.8,
                f"{v:.2f}%", ha="center", fontsize=11, fontweight="bold")
    ax.set_title(title, fontsize=13, pad=10)
    ax.set_xlabel("Train : Test Split")
    ax.set_ylabel("Score (%)")
    ax.set_ylim(0, max(vals or [0]) + 15)
    ax.axhline(np.mean(vals), color="gray", linestyle="--", lw=1, label=f"Avg: {np.mean(vals):.1f}%")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("02_f1_precision_recall_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
#Accuracy and Error Rate
acc_vals = [r["Accuracy"]*100  for r in metrics_rows]
err_vals = [r["Error Rate"]*100 for r in metrics_rows]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("SARIMA — Accuracy & Error Rate", fontsize=14, fontweight="bold")

# Accuracy
bars = axes[0].bar(split_labels, acc_vals, color=BLUE, alpha=0.88,
                   width=0.5, edgecolor="white")
for bar, v in zip(bars, acc_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                 f"{v:.2f}%", ha="center", fontsize=12, fontweight="bold")
axes[0].set_title("Accuracy (%)", fontsize=13, pad=10)
axes[0].set_xlabel("Train : Test Split")
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_ylim(0, 110)
axes[0].axhline(np.mean(acc_vals), color="gray", linestyle="--", lw=1,
                label=f"Avg: {np.mean(acc_vals):.1f}%")
axes[0].legend()

# Error Rate
bars = axes[1].bar(split_labels, err_vals, color=RED, alpha=0.88,
                   width=0.5, edgecolor="white")
for bar, v in zip(bars, err_vals):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f"{v:.2f}%", ha="center", fontsize=12, fontweight="bold")
axes[1].set_title("Error Rate  (%) = 1 − Accuracy", fontsize=13, pad=10)
axes[1].set_xlabel("Train : Test Split")
axes[1].set_ylabel("Error Rate (%)")
axes[1].set_ylim(0, max(err_vals or [5]) + 5)
axes[1].axhline(np.mean(err_vals), color="gray", linestyle="--", lw=1,
                label=f"Avg: {np.mean(err_vals):.1f}%")
axes[1].legend()

plt.tight_layout()
plt.savefig("03_accuracy_errorrate.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
#ROC Curve & AUC (all 4 splits)
colors_roc = [BLUE, ORANGE, GREEN, RED]

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0,1],[0,1], "k--", lw=1, label="Random classifier (AUC=0.50)")

for (label, r), color in zip(results.items(), colors_roc):
    yt, ys = r["y_true"], r["y_score"]
    try:
        fpr, tpr, _ = roc_curve(yt, ys)
        auc_val     = roc_auc_score(yt, ys)
        ax.plot(fpr, tpr, lw=2, color=color,
                label=f"Split {label}  (AUC = {auc_val:.4f})")
    except Exception as e:
        print(f"  ROC skipped for {label}: {e}")

ax.set_title("SARIMA — ROC Curve & Area Under Curve (AUC)", fontsize=14, fontweight="bold")
ax.set_xlabel("False Positive Rate (1 − Specificity)")
ax.set_ylabel("True Positive Rate (Sensitivity)")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.legend(loc="lower right", fontsize=9)
ax.fill_between([0,1],[0,1], alpha=0.05, color="gray")

plt.tight_layout()
plt.savefig("04_roc_auc_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
#FULL SUMMARY(all metrics, all splits)
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle("SARIMA Model — Complete Results Dashboard\n"
             f"Dataset: Rainfall in India | Subdivision: {TARGET_SUB}",
             fontsize=15, fontweight="bold", y=1.01)

metric_keys  = ["Accuracy","Precision","Recall / Sensitivity",
                "F1-Score","AUC","Error Rate"]
metric_labels= ["Accuracy (%)","Precision (%)","Recall / Sensitivity (%)",
                "F1-Score (%)","AUC","Error Rate (%)"]
panel_colors = [BLUE, ORANGE, GREEN, "#9B5DE5", "#F72585", RED]
scale_auc    = [False,False,False,False,True,False]

all_axes = axes.flatten()

for i, (key, lbl, col, is_auc) in enumerate(
        zip(metric_keys, metric_labels, panel_colors, scale_auc)):
    ax = all_axes[i]
    vals = [r[key]*(1 if is_auc else 100) for r in metrics_rows]
    bars = ax.bar(split_labels, vals, color=col, alpha=0.85,
                  width=0.5, edgecolor="white")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height() + (0.005 if is_auc else 0.4),
                f"{v:.4f}" if is_auc else f"{v:.2f}%",
                ha="center", fontsize=9, fontweight="bold")
    ax.set_title(lbl, fontsize=11, pad=8)
    ax.set_ylim(0, (max(vals or [1])+(0.1 if is_auc else 12)))
    ax.axhline(np.mean(vals), color="gray", linestyle="--", lw=1)
    ax.tick_params(labelsize=9)

# 7th panel: confusion matrix for 80:20
ax7 = all_axes[6]
r80 = results["80:20"]
cm  = confusion_matrix(r80["y_true"], r80["y_pred"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax7,
            xticklabels=["Normal","Anomaly"],
            yticklabels=["Normal","Anomaly"],
            annot_kws={"size":12, "weight":"bold"}, cbar=False)
ax7.set_title("Confusion Matrix (80:20)", fontsize=11, pad=8)
ax7.set_xlabel("Predicted"); ax7.set_ylabel("Actual")

# 8th panel: ROC for 80:20
ax8 = all_axes[7]
yt80, ys80 = r80["y_true"], r80["y_score"]
try:
    fpr80, tpr80, _ = roc_curve(yt80, ys80)
    auc80 = roc_auc_score(yt80, ys80)
    ax8.plot(fpr80, tpr80, lw=2, color=BLUE, label=f"AUC = {auc80:.4f}")
    ax8.plot([0,1],[0,1],"k--",lw=1)
    ax8.fill_between(fpr80, tpr80, alpha=0.1, color=BLUE)
    ax8.set_title("ROC Curve (80:20 Split)", fontsize=11, pad=8)
    ax8.set_xlabel("FPR"); ax8.set_ylabel("TPR")
    ax8.legend(fontsize=9)
except Exception:
    ax8.text(0.5,0.5,"AUC N/A",ha="center",va="center",transform=ax8.transAxes)

plt.tight_layout()
plt.savefig("05_summary_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
#5 Test Case Validation (80:20 split)
r80      = results["80:20"]
ts_test  = r80["ts_test"]
forecast = r80["forecast"]
y_true80 = r80["y_true"]
y_pred80 = r80["y_pred"]
res_z80  = r80["res_z"]
n_test   = len(ts_test)

indices = [0, n_test//4, n_test//2, 3*n_test//4, n_test-1]

print("="*80)
print("5 TEST CASE VALIDATION — SARIMA (80:20 Split)")
print("="*80)
print(f"{'Case':<6}{'Date':<12}{'Actual RF':>12}{'Forecast':>12}"
      f"{'Residual':>12}{'Res Z':>8}{'True':>10}{'Predicted':>11}{'Correct':>9}")
print("-"*80)

val_rows = []
for i, idx in enumerate(indices, 1):
    date     = ts_test.index[idx].strftime("%b %Y")
    actual   = round(float(ts_test.iloc[idx]), 2)
    pred_val = round(float(forecast.iloc[idx]), 2)
    residual = round(actual - pred_val, 2)
    rz       = round(float(res_z80[idx]), 3)
    t_lbl    = "ANOMALY" if y_true80[idx] else "normal"
    p_lbl    = "ANOMALY" if y_pred80[idx] else "normal"
    correct  = "✓ YES" if y_true80[idx] == y_pred80[idx] else "✗ NO"
    val_rows.append([i, date, actual, pred_val, residual, rz, t_lbl, p_lbl, correct])
    print(f"  {i:<4}{date:<12}{actual:>12.2f}{pred_val:>12.2f}"
          f"{residual:>12.2f}{rz:>8.3f}{t_lbl:>10}{p_lbl:>11}{correct:>9}")

print("-"*80)
correct_n = sum(1 for r in val_rows if "YES" in r[8])
print(f"\n  Validation accuracy: {correct_n}/5 correct  ({correct_n*20}%)")
print("="*80)

In [ ]:
#Forecast vs Actual Plot (80:20 split)
r80 = results["80:20"]
ts_test_vals  = r80["ts_test"].values
forecast_vals = r80["forecast"].values
n = len(ts_test_vals)
months = list(range(n))

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle(f"SARIMA — Forecast vs Actual Rainfall\nSubdivision: {TARGET_SUB} | 80:20 Split",
             fontsize=14, fontweight="bold")

# Top: rainfall values
axes[0].plot(months, ts_test_vals, color=BLUE, lw=1.5, label="Actual Rainfall")
axes[0].plot(months, forecast_vals, color=ORANGE, lw=1.5,
             linestyle="--", label="SARIMA Forecast")
axes[0].fill_between(months, ts_test_vals, forecast_vals,
                     alpha=0.1, color="gray", label="Residual area")
axes[0].set_ylabel("Rainfall (mm)")
axes[0].legend()

# Middle: residual Z-scores with threshold lines
rz = r80["res_z"]
axes[1].bar(months, rz, color=[RED if abs(z)>2 else BLUE for z in rz], alpha=0.7)
axes[1].axhline( 2, color=RED, linestyle="--", lw=1.5, label="+2.0 threshold")
axes[1].axhline(-2, color=RED, linestyle="--", lw=1.5, label="−2.0 threshold")
axes[1].set_ylabel("Residual Z-Score")
axes[1].legend(fontsize=9)

# Bottom: predicted vs true anomaly flags
axes[2].step(months, r80["y_true"], where="mid",
             color=BLUE, lw=2, label="True Anomaly")
axes[2].step(months, r80["y_pred"], where="mid",
             color=ORANGE, lw=2, linestyle="--", label="Predicted Anomaly")
for idx in indices:
    axes[2].axvline(idx, color=GREEN, linestyle=":", lw=1.2, alpha=0.8)
axes[2].set_ylabel("Anomaly (1=Yes)")
axes[2].set_xlabel("Test Period (months)")
axes[2].set_yticks([0,1])
axes[2].legend(fontsize=9)
axes[2].text(0.01, 0.85, "Green dashed = 5 validation cases",
             transform=axes[2].transAxes, fontsize=8, color=GREEN)

plt.tight_layout()
plt.savefig("06_forecast_vs_actual.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
#Final summary
print()
print("╔══════════════════════════════════════════════════════════════╗")
print("║          SARIMA MODEL — FINAL RESULTS SUMMARY               ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Model      : SARIMA(1,1,1)(1,1,1,12)                       ║")
print(f"║  Dataset    : Rainfall in India (1901–2015) Datase          ║")
print(f"║  Subdivision: {TARGET_SUB:<46} ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"{'║  Split':<12}{'Accuracy':>10}{'Precision':>11}{'Recall':>9}"
      f"{'F1':>8}{'AUC':>9}{'ErrRate':>10}  ║")
print("╠══════════════════════════════════════════════════════════════╣")
for r in metrics_rows:
    print(f"║  {r['Split']:<10}{r['Accuracy']*100:>9.2f}%"
          f"{r['Precision']*100:>10.2f}%"
          f"{r['Recall / Sensitivity']*100:>8.2f}%"
          f"{r['F1-Score']*100:>7.2f}%"
          f"{r['AUC']:>9.4f}"
          f"{r['Error Rate']*100:>9.2f}%  ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  5-Case Validation (80:20): {correct_n}/5 correct = {correct_n*20}%              ║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║  Files saved:                                                ║")
print("║    01_confusion_matrix.png       (Screenshot 1)              ║")
print("║    02_f1_precision_recall_sensitivity.png  (Screenshots 2-5) ║")
print("║    03_accuracy_errorrate.png     (Screenshots 6 & 8)         ║")
print("║    04_roc_auc_curve.png          (Screenshot 7)              ║")
print("║    05_summary_dashboard.png      (Full dashboard)            ║")
print("║    06_forecast_vs_actual.png     (Time series plot)          ║")
print("╚══════════════════════════════════════════════════════════════╝")

In [ ]:

from google.colab import files
import os

png_files = [f for f in os.listdir(".") if f.endswith(".png")]
png_files.sort()
print(f"Downloading {len(png_files)} files...")
for f in png_files:
    files.download(f)
    print(f"  ↓ {f}")